# 05a – Baseline en een snel eerste model (2024)

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Bepalen wat "goed" is: hoe ver geraak je met gewone gemiddelden? Daarna een eerste, ongetuned model zodat de deployment (API + webpagina) al kan starten terwijl we verder tunen. |
| **Input** | `data/2024/model_table.parquet` (uit `04_prepare_data`) |
| **Output** | `models/2024/metrics/baseline.json`, `models/2024/metrics/quick_hgb.json`, `models/2024/predictions/*.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

> **Studie op 2024 (modelselectie).** Dit notebook werkt op één jaar (2024) om snel en grondig te kiezen welke techniek het best werkt. De gekozen techniek wordt daarna op alle data (2013–2026) getraind: zie `02b_eda_all_years`, `04_prepare_data` en `07`–`09`.

## Hoe meten we in alle modelnotebooks?
- **Target:** aantal vertrekken per zone per uur (een telling ≥ 0).
- **Testset:** de laatste 7 dagen van elke maand (zie `04_prepare_data`). Die gebruiken we per model maar **één keer**, op het einde.
- **Validatie tijdens het kiezen/tunen:** 5-voudige `GroupKFold` op de trainset, met de **kalenderweek** als groep. Zo komen uren uit dezelfde week nooit tegelijk in train en validatie terecht (naburige uren lijken sterk op elkaar: dat zou de score te mooi maken).
- **Metrics:**
  - **MAE** (primair): "gemiddeld zitten we x vertrekken per uur naast". Makkelijk uit te leggen aan een planner.
  - **RMSE**: straft grote fouten zwaarder (bv. een gemiste spits).
  - **R²**: welk deel van de variatie verklaren we?
  - **Poisson-deviance**: de "natuurlijke" foutmaat voor tellingen. Een fout van 5 bij een verwachting van 2 is erger dan een fout van 5 bij een verwachting van 200.

## 0. Imports en data

In [1]:
import json
import subprocess
import sys

import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

import citibike as cb

table = cb.load_model_table(cb.DATA_2024 / "model_table.parquet")
X_train, y_train, g_train, X_test, y_test = cb.train_test(table, cb.FEATURES_2024)
print(f"train: {len(X_train):,} zone-uren in {g_train.nunique()} weken | test: {len(X_test):,} zone-uren")
X_train.head()

train: 203,040 zone-uren in 50 weken | test: 60,480 zone-uren


,zone,hour,weekday,month,is_weekend,is_holiday,temperature_c,precipitation_mm,wind_kmh
0,0,0,0,1,0,1,1.8,0.1,7.4
1,0,1,0,1,0,1,1.3,0.0,8.7
2,0,2,0,1,0,1,2.4,0.0,10.4
3,0,3,0,1,0,1,2.7,0.0,9.2
4,0,4,0,1,0,1,2.7,0.0,7.9


## 1. Drie baselines met gemiddelden
Een model is pas nuttig als het beter doet dan wat een planner met Excel zou doen. We proberen drie steeds fijnere gemiddelden uit de trainset:

| Baseline | Voorspelling |
|---|---|
| globaal | het gemiddelde over alle zones en uren |
| per zone | het gemiddelde van die zone |
| **per zone × uur × weekdag/weekend** | "hoeveel vertrekken er gemiddeld in deze zone om 8u op een weekdag?" |

De laatste is de echte lat: hij kent het dagpatroon van elke zone al, alleen het weer en het seizoen niet.

In [2]:
baselines = {
    "globaal gemiddelde": cb.GroupMeanRegressor(keys=()),
    "gemiddelde per zone": cb.GroupMeanRegressor(keys=("zone",)),
    "gemiddelde per zone x uur x weekend": cb.GroupMeanRegressor(keys=("zone", "hour", "is_weekend")),
}
rows = []
for name, model in baselines.items():
    cv = cb.cv_scores(model, X_train, y_train, g_train)
    rows.append({"baseline": name, "cv_mae": cv["cv_mae_mean"], **{f"cv_fold_{i}": m for i, m in enumerate(cv["cv_mae"])}})
pd.DataFrame(rows).set_index("baseline")

,cv_mae,cv_fold_0,cv_fold_1,cv_fold_2,cv_fold_3,cv_fold_4
baseline,,,,,,
globaal gemiddelde,138.4346,140.0145,136.2310,142.4472,132.3894,141.0907
gemiddelde per zone,116.9228,119.0662,113.8980,123.0739,108.3841,120.1920
gemiddelde per zone x uur x weekend,59.1855,60.5741,58.4869,69.9890,52.3978,54.4795


**Interpretatie:**
- Het **globale gemiddelde** zit gemiddeld 138 vertrekken per uur naast: geen verrassing, want het weet niets.
- **Per zone** helpt weinig (117): het verschil tussen zones is kleiner dan het verschil tussen dag en nacht.
- **Zone × uur × weekend** halveert de fout (59). Het dagritme per plaats is dus de grootste bron van informatie, precies wat de EDA (02 §7 en §9) en H2 (03) voorspelden.
- De fout verschilt per fold (52 tot 70): sommige weken zijn moeilijker (zomer met meer ritten, of weken met veel regen). Daarom vergelijken we modellen altijd **op dezelfde folds**.

## 2. De beste baseline op de testset
We nemen de fijnste baseline als referentie voor alle volgende modellen en meten hem één keer op de testset.

In [3]:
baseline = baselines["gemiddelde per zone x uur x weekend"].fit(X_train, y_train)
pred = baseline.predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
cv = cb.cv_scores(baselines["gemiddelde per zone x uur x weekend"], X_train, y_train, g_train)
metrics = {"notebook": "05a_baseline", "model": "Baseline: gemiddelde per zone x uur x weekend",
           **cv, **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("baseline", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred),
                cb.METRICS_2024, cb.PRED_2024)
test_metrics

{'mae': 59.2751, 'rmse': 96.4412, 'r2': 0.7235, 'poisson_deviance': 34.1876}

## 3. Snel eerste model: gradient boosting met standaardinstellingen
Om de deployment al te kunnen bouwen, hebben we nu een "echt" model nodig dat het weer en de kalender gebruikt. We kiezen `HistGradientBoostingRegressor` van scikit-learn, **zonder te tunen**:
- **boomgebaseerd**: vindt zelf de interacties (zone × uur × weekend × regen) die een lineair model met de hand moet krijgen;
- **`loss="poisson"`**: voorspelt tellingen (nooit negatief, fout relatief tot de verwachting);
- **zone als categorische variabele** (`categorical_features`): zone 7 is niet "groter" dan zone 3;
- snel op 200.000+ rijen, en scikit-learn staat al op de server (geen extra bibliotheek voor de API).

In [4]:
quick = HistGradientBoostingRegressor(loss="poisson", categorical_features=[cb.FEATURES_2024.index("zone")],
                                      random_state=cb.SEED)
cv = cb.cv_scores(quick, X_train, y_train, g_train)
quick.fit(X_train, y_train)
pred = quick.predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "05a_baseline", "model": "HistGradientBoosting (Poisson, standaard)",
           "params": {}, **cv, **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("quick_hgb", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred),
                cb.METRICS_2024, cb.PRED_2024)
pd.DataFrame({"baseline": json.loads((cb.METRICS_2024 / "baseline.json").read_text()),
              "snel model": metrics}).loc[["cv_mae_mean", "test_mae", "test_rmse", "test_r2", "test_poisson_deviance"]]

,baseline,snel model
cv_mae_mean,59.1855,27.5663
test_mae,59.2751,30.8826
test_rmse,96.4412,57.1495
test_r2,0.7235,0.9029
test_poisson_deviance,34.1876,13.3123


**Interpretatie:** het ongetunede boosting-model **halveert de fout opnieuw**: MAE 31 tegenover 59 voor de beste baseline, en R² 0,90 tegenover 0,72. Wat het extra weet ten opzichte van de baseline: het weer, de maand (seizoen), de weekdag (vrijdag ≠ dinsdag) en feestdagen. Dat het weer zoveel uitmaakt, klopt met H1 (03).

De testfout (30,9) is iets hoger dan de CV-fout (27,6). De testweken bevatten enkele moeilijke dagen: Thanksgiving (28/11), Kerstmis en oudejaar, en Memorial Day (27/5) vallen allemaal in de laatste week van hun maand. In `06` kijken we of de grootste fouten daar zitten.

## 4. Eerste versie van de deployment
In de 2024-studie startte de API hier met dit snelle model, zodat de webpagina al gebouwd kon worden. Het gedeployde model wordt nu getraind op alle data: zie `07_baseline_all_data` en `08_hgb_all_data`.

## Samenvatting
| Model | CV-MAE | Test-MAE | Test-R² |
|---|---|---|---|
| globaal gemiddelde | 138,4 | | |
| gemiddelde per zone | 116,9 | | |
| **baseline**: gemiddelde per zone × uur × weekend | 59,2 | 59,3 | 0,72 |
| **snel model**: HistGradientBoosting (Poisson, standaard) | 27,6 | 30,9 | 0,90 |

- De lat voor alle volgende modellen: **MAE 59** (baseline). Een model dat daar niet ver onder zit, voegt niets toe.
- Dit snelle model was de eerste versie in de API, zodat de webpagina gebouwd en getest kon worden terwijl we verder tunden (05b tot 05e).